# SSD 配對交易滾動回測系統 - 交易邏輯與公式說明

本系統是一個基於 **SSD (Sum of Squared Differences)** 與 **Z-Score** 的配對交易 (Pairs Trading) 回測框架。其核心精神是透過形成期尋找歷史走勢高度相似的股票對，並在交易期監控其價差，當價差發生極端偏離時進行反向操作，押注其均值回歸 (Mean Reversion)。

本系統採用「**滾動視窗 (Rolling Window)**」架構，定期重新篩選配對，以適應市場動態變化。

In [1]:
"""
SSD 配對交易滾動回測系統 (交易明細版)
核心功能：基於 SSD (Sum of Squared Differences) 與 Z-Score 的配對交易回測。
針對大型網格搜索進行效能最佳化，僅輸出詳細交易紀錄。
"""

import sqlite3
import warnings
import itertools
from datetime import datetime
from pathlib import Path
from dataclasses import dataclass

import numpy as np
import pandas as pd
import scipy.spatial.distance as ssd

# 忽略不必要的 Pandas/Numpy 警告以保持輸出整潔
warnings.filterwarnings("ignore")


# 一、形成期 (Formation Period)：配對篩選

形成期的主要目標是在給定的歷史視窗內，找出累積回報走勢最相似（即 SSD 距離最小）的股票組合。

## 1. 價格正規化 (Price Normalization)

為了消除不同股票價格絕對水準的差異，累積總回報指數 (Cumulative Total Returns Index) 被計算出來，並在首日設為 1.0。

**公式**：

對於股票 $i$ 在時間 $t$ 的價格 $P_{i,t}$，首日價格為 $P_{i,0}$：

- 累積總回報指數 (Normalized Cumulative Price)：  
  $$ P_{i,t}^* = \frac{P_{i,t}}{P_{i,0}} $$

## 2. 距離計算與配對排序 (SSD Computation)

系統計算產業內任兩檔股票 $A$ 與 $B$ 之間累積回報指數序列的歐氏距離平方和 (SSD)。

**公式**：
$$
SSD(A, B) = \sum_{t=1}^{T} (P_{A,t}^* - P_{B,t}^*)^2
$$

系統最終會選取 SSD 值最小的前 **Top_N** 組配對進入交易期。

## 3. 價差與信號估計 (Spread Parameters)

距離法不需要動態避險比例 (Beta 隱含為 1.0)。

**公式**：

- 形成期價差序列 (Spread)：  
  $$Spread_t = P_{A,t}^* - P_{B,t}^*$$

- 價差參數：  
  均值：$\mu_{spread} = Mean(Spread)$  
  標準差：$\sigma_{spread} = Std(Spread)$


In [2]:
# ══════════════════════════════════════════════════════════════════════════════
# Class 1：Formation（形成期模組）
# ══════════════════════════════════════════════════════════════════════════════
class Formation:
    """
    負責在形成期 (Formation Period) 篩選最佳配對。
    透過計算累積總回報指數 (Cumulative Total Returns Index) 的 SSD 來尋找走勢相近的股票對。
    累積總回報指數第一天價格歸一化為 1.0 (完全對應 Gatev et al. 2006 論文)。
    """
    def __init__(self, price_df: pd.DataFrame, form_start: str, form_end: str, top_n: int = 20, sector_mapping: dict = None, min_tickers_for_pairing: int = 2):
        self.price_df = price_df.copy()
        self.form_start = form_start
        self.form_end = form_end
        self.top_n = top_n
        self.sector_mapping = sector_mapping or {}
        self.min_tickers_for_pairing = min_tickers_for_pairing

        self.normalized_df: pd.DataFrame = pd.DataFrame()
        self.first_day_prices: pd.Series = pd.Series(dtype=float)
        self.selected_pairs: pd.DataFrame = pd.DataFrame()

    def normalize_prices(self) -> pd.DataFrame:
        """將價格正規化為累積總回報指數 (首日價格設為 1.0)"""
        self.first_day_prices = self.price_df.iloc[0]
        # 防範分母為 0，若首日價格 <= 0 則設為 1.0
        safe_first_prices = np.where(self.first_day_prices > 1e-8, self.first_day_prices, 1.0)
        self.normalized_df = self.price_df / safe_first_prices
        return self.normalized_df

    def compute_ssd(self) -> pd.DataFrame:
        """計算產業內所有可能配對的 SSD (Sum of Squared Differences)"""
        if self.normalized_df.empty:
            self.normalize_prices()

        tickers = self.normalized_df.columns.tolist()
        ssd_records = []

        # 根據產業分類分組
        sector_groups = {}
        if self.sector_mapping:
            for ticker in tickers:
                sector = self.sector_mapping.get(ticker, "Unknown")
                sector_groups.setdefault(sector, []).append(ticker)
        else:
            sector_groups["All_Market"] = tickers

        skipped_unknown_count = 0
        for sector, sector_tickers in sector_groups.items():
            if sector == "Unknown":
                skipped_unknown_count = len(sector_tickers)
                continue
            
            if len(sector_tickers) < self.min_tickers_for_pairing: 
                continue

            # 向量化最佳化：使用 scipy 進行快速的兩兩配對距離計算
            norm_vals = self.normalized_df[sector_tickers].values.T
            
            # 使用 squared euclidean 計算 SSD
            ssd_matrix = ssd.pdist(norm_vals, metric='sqeuclidean')
            
            # 將 pdist 的 1D 陣列轉換為 Pair 組合
            # 距離法中 Beta 恆定為 1.0 (等權重 dollar weighting)
            idx = 0
            for i in range(len(sector_tickers)):
                ticker_b = sector_tickers[i]
                x_val = norm_vals[i]
                
                for j in range(i + 1, len(sector_tickers)):
                    ticker_a = sector_tickers[j]
                    y_val = norm_vals[j]
                    
                    ssd_value = ssd_matrix[idx]
                    idx += 1
                    
                    beta = 1.0 # 論文無 Beta 估計，等同於 1.0
                    spread = y_val - beta * x_val
                    spread_mean = np.mean(spread)
                    spread_std = np.std(spread, ddof=1) if len(spread) > 1 else 0.0
                    
                    ssd_records.append({
                        "Form_Start": self.form_start, "Form_End": self.form_end,
                        "Sector": sector, "Ticker_A": ticker_a, "Ticker_B": ticker_b,
                        "SSD": round(ssd_value, 6), "Hedge_Ratio": round(beta, 4),
                        "Spread_Mean": round(spread_mean, 6),
                        "Spread_Std": round(spread_std, 6)
                    })

        if skipped_unknown_count > 0:
            print(f"  [Formation] 跳過 {skipped_unknown_count} 支未分類 (Unknown) 股票。")

        if not ssd_records: 
            return pd.DataFrame()
            
        return pd.DataFrame(ssd_records).sort_values("SSD").reset_index(drop=True)

    def select_pairs(self) -> pd.DataFrame:
        """選出 SSD 最小的前 N 組配對"""
        ssd_df = self.compute_ssd()
        if ssd_df.empty:
            self.selected_pairs = pd.DataFrame()
            return self.selected_pairs

        selected = ssd_df.head(self.top_n).copy()
        selected["Rank"] = range(1, len(selected) + 1)

        first_price_a_list, first_price_b_list = [], []
        for _, row in selected.iterrows():
            first_price_a_list.append(self.price_df[row["Ticker_A"]].iloc[0])
            first_price_b_list.append(self.price_df[row["Ticker_B"]].iloc[0])
            
        selected["First_Price_A"] = first_price_a_list
        selected["First_Price_B"] = first_price_b_list

        self.selected_pairs = selected
        return self.selected_pairs

    def run(self) -> pd.DataFrame:
        """執行形成期流程並回傳選定的配對"""
        self.normalize_prices()
        self.select_pairs()
        return self.selected_pairs

# 二、交易期 (Trading Period)：價差監控與交易執行

交易期接收形成期選出的配對，並根據其價格變動產生 Z-Score 訊號來觸發進出場。

## 1. 交易期價格正規化

交易期的價格 $P_t$，必須使用形成期所第一天的價格 $P_{i,0}$ 進行相同的正規化處理。

$$
P'_{A,t} = \frac{P_{A,t}}{P_{A,0}} \\
P'_{B,t} = \frac{P_{B,t}}{P_{B,0}}
$$

## 2. Z-Score 計算模式

$$
Spread'_t = P'_{A,t} - P'_{B,t} \\
Z_t = \frac{Spread'_t - \mu_{spread}}{\sigma_{spread}}
$$


In [3]:
# ══════════════════════════════════════════════════════════════════════════════
# Class 2：Trading（交易期模組）
# ══════════════════════════════════════════════════════════════════════════════
@dataclass(slots=True)
class PairState:
    """單一配對在模擬過程中的內部狀態 (使用 slots 降低記憶體開銷)"""
    position: int = 0
    shares_a: float = 0.0
    shares_b: float = 0.0
    entry_price_a: float = 0.0
    entry_price_b: float = 0.0
    realized_pnl: float = 0.0
    trade_entry_fee: float = 0.0
    days_held: int = 0
    is_stopped: bool = False
    cooldown_dir: int = 0
    prev_total_pnl: float = 0.0

class Trading:
    """負責在交易期 (Trading Period) 模擬配對交易並產生詳細交易紀錄"""
    def __init__(self, price_df: pd.DataFrame, trade_dates: pd.DatetimeIndex, selected_pairs: pd.DataFrame, capital_per_pair: float, 
                 fee_rate: float, slippage_rate: float, stop_loss_pct: float, entry_z: float, exit_z: float, zscore_window: int, allow_reentry: bool = False,
                 zscore_clip: float = 10.0, min_spread_std: float = 1e-6):
        self.price_df = price_df.copy()
        self.trade_dates = trade_dates
        self.selected_pairs = selected_pairs
        self.capital_per_pair = capital_per_pair
        
        self.friction_rate = fee_rate + slippage_rate
        self.stop_loss_pct = stop_loss_pct
        self.entry_z = entry_z
        self.exit_z  = exit_z
        self.zscore_window = zscore_window
        self.allow_reentry = allow_reentry  
        self.zscore_clip = zscore_clip
        self.min_spread_std = min_spread_std

        self.period_pnl: float = 0.0

    def _execute_entry(self, state: PairState, z: float, p_a: float, p_b: float) -> tuple[bool, float]:
        """處理進場邏輯與資金分配 (1:1 等額市值中性 Dollar Weighting)"""
        # 完全對應論文，1:1 等額資金配置
        v_a = self.capital_per_pair * 0.5
        v_b = self.capital_per_pair * 0.5
        
        # Z-Score 超過門檻且不在冷卻期時進場
        if z > self.entry_z and state.cooldown_dir != -1:
            state.position = -1
            state.shares_a = -v_a / p_a
            state.shares_b = v_b / p_b
        elif z < -self.entry_z and state.cooldown_dir != 1:
            state.position = +1
            state.shares_a = v_a / p_a
            state.shares_b = -v_b / p_b
        else:
            return False, 0.0

        state.entry_price_a = p_a
        state.entry_price_b = p_b
        state.trade_entry_fee = (abs(state.shares_a) * p_a + abs(state.shares_b) * p_b) * self.friction_rate
        state.days_held = 0
        return True, -state.trade_entry_fee

    def _execute_close(self, state: PairState, current_trade_pnl: float, stop_loss: bool = False):
        """處理平倉與停損邏輯"""
        state.realized_pnl += current_trade_pnl 
        
        if stop_loss:
            if not self.allow_reentry:
                state.is_stopped = True
            else:
                state.cooldown_dir = state.position 
        else:
            state.cooldown_dir = 0  
                
        state.position = 0
        
        state.shares_a = 0.0
        state.shares_b = 0.0
        state.entry_price_a = 0.0
        state.entry_price_b = 0.0
        state.trade_entry_fee = 0.0

    def _simulate_pair(self, period_start: str, period_end: str, sector: str, ticker_a: str, ticker_b: str, pair_rank: int, hedge_ratio: float, 
                       form_spread_mean: float, form_spread_std: float, first_price_a: float, first_price_b: float) -> pd.DataFrame:
        if ticker_a not in self.price_df.columns or ticker_b not in self.price_df.columns: return pd.DataFrame()

        price_a, price_b = self.price_df[ticker_a].dropna(), self.price_df[ticker_b].dropna()
        common_idx = price_a.index.intersection(price_b.index)
        price_a, price_b = price_a.loc[common_idx], price_b.loc[common_idx]

        if len(price_a) < 5: return pd.DataFrame()

        # 累積總回報指數正規化 (第一天設為 1.0，以形成期首日價格 first_price 為分母)
        norm_p_a = price_a / (first_price_a if first_price_a > 1e-8 else 1.0)
        norm_p_b = price_b / (first_price_b if first_price_b > 1e-8 else 1.0)
        
        # 計算 Z-Score
        # 距離法中 Beta 鎖定為 1.0
        if self.zscore_window == 0:
            spread = norm_p_a - norm_p_b
            safe_std = max(form_spread_std, self.min_spread_std)
            zscore = np.clip((spread - form_spread_mean) / safe_std, -self.zscore_clip, self.zscore_clip)
            beta_series = pd.Series(1.0, index=common_idx)
        else:
            # 滾動模式下亦鎖定 Beta = 1.0，但支持滾動去中心化
            roll_mean_a = norm_p_a.rolling(window=self.zscore_window).mean()
            roll_mean_b = norm_p_b.rolling(window=self.zscore_window).mean()
            roll_alpha = roll_mean_a - roll_mean_b
            spread = norm_p_a - roll_alpha - norm_p_b
            
            roll_std = (norm_p_a - norm_p_b).rolling(window=self.zscore_window).std()
            if (roll_std < self.min_spread_std * 10).mean() > 0.5:
                return pd.DataFrame()
            
            safe_std = np.maximum(roll_std, self.min_spread_std)
            zscore = np.clip(spread / safe_std, -self.zscore_clip, self.zscore_clip)
            beta_series = pd.Series(1.0, index=common_idx)

        valid_idx = common_idx.intersection(self.trade_dates)
        if len(valid_idx) == 0: return pd.DataFrame()
        
        price_a = price_a.loc[valid_idx]
        price_b = price_b.loc[valid_idx]
        zscore = zscore.loc[valid_idx]
        beta_series = beta_series.loc[valid_idx]

        dates_arr = valid_idx
        zscore_arr = zscore.values
        pa_arr = price_a.values
        pb_arr = price_b.values
        beta_arr = beta_series.values

        base_log = {
            "Period_Start": period_start, "Period_End": period_end,
            "Sector": sector, "Pair_Rank": pair_rank,
            "Ticker_A": ticker_a, "Ticker_B": ticker_b,
            "First_Price_A": first_price_a, "First_Price_B": first_price_b
        }

        state = PairState()
        
        out_dates, out_pa, out_pb = [], [], []
        out_hr, out_z, out_pos = [], [], []
        out_unrealized, out_realized, out_cum = [], [], []
        out_status, out_trade_pnl, out_days, out_delta = [], [], [], []

        for i in range(len(dates_arr)):
            date = dates_arr[i]
            z = 0.0 if np.isnan(zscore_arr[i]) else zscore_arr[i]
            p_a, p_b = pa_arr[i], pb_arr[i]
            c_beta = 1.0

            unrealized_pnl = 0.0
            closed_trade_pnl = 0.0 
            daily_delta = 0.0
            current_status = "HOLD_CASH"

            if state.is_stopped:
                out_dates.append(date)
                out_pa.append(round(p_a, 4))
                out_pb.append(round(p_b, 4))
                out_hr.append(1.0)
                out_z.append(round(float(z), 4))
                out_pos.append(0)
                out_unrealized.append(0.0)
                out_realized.append(round(float(state.realized_pnl), 4))
                out_cum.append(round(float(state.realized_pnl), 4))
                out_status.append("STOPPED")
                out_trade_pnl.append(0.0)
                out_days.append(0)
                out_delta.append(0.0)
                continue

            if state.cooldown_dir == -1 and z <= self.exit_z:
                state.cooldown_dir = 0
            elif state.cooldown_dir == 1 and z >= -self.exit_z:
                state.cooldown_dir = 0

            if state.position != 0:
                state.days_held += 1
                raw_unrealized = state.shares_a * (p_a - state.entry_price_a) + state.shares_b * (p_b - state.entry_price_b)
                exit_fee_est = (abs(state.shares_a)*p_a + abs(state.shares_b)*p_b) * self.friction_rate
                
                current_trade_pnl = raw_unrealized - state.trade_entry_fee - exit_fee_est
                
                if self.stop_loss_pct > 0 and (-current_trade_pnl / self.capital_per_pair) >= self.stop_loss_pct:
                    self._execute_close(state, current_trade_pnl, stop_loss=True)
                    closed_trade_pnl = current_trade_pnl
                    current_status = "STOP_LOSS_TRIGGERED"
                else:
                    is_exit_short = (state.position == -1) and (z <= self.exit_z)  
                    is_exit_long  = (state.position == 1)  and (z >= -self.exit_z)  
                    
                    if is_exit_short or is_exit_long:
                        self._execute_close(state, current_trade_pnl, stop_loss=False)
                        closed_trade_pnl = current_trade_pnl
                        current_status = "EXIT"
                    else:
                        unrealized_pnl = current_trade_pnl 
                        current_status = "HOLDING"
            else: 
                if abs(z) > self.entry_z:
                    entered, unrealized_pnl = self._execute_entry(state, z, p_a, p_b)
                    if entered:
                        current_status = "ENTER_SHORT_A" if state.position == -1 else "ENTER_LONG_A"
                    else:
                        current_status = "HOLD_CASH (COOLDOWN)"
                else:
                    current_status = "HOLD_CASH"

            cumulative_pnl = state.realized_pnl + unrealized_pnl
            daily_delta = cumulative_pnl - state.prev_total_pnl
            state.prev_total_pnl = cumulative_pnl
            
            out_dates.append(date)
            out_pa.append(round(p_a, 4))
            out_pb.append(round(p_b, 4))
            out_hr.append(1.0)
            out_z.append(round(float(z), 4))
            out_pos.append(state.position)
            out_unrealized.append(round(float(unrealized_pnl), 4))
            out_realized.append(round(float(state.realized_pnl), 4))
            out_cum.append(round(float(cumulative_pnl), 4))
            out_status.append(current_status)
            out_trade_pnl.append(round(float(closed_trade_pnl), 4))
            out_days.append(state.days_held)
            out_delta.append(round(float(daily_delta), 4))

            if current_status in ["STOP_LOSS_TRIGGERED", "EXIT"]:
                state.days_held = 0 

            if state.is_stopped and i < len(dates_arr) - 1:
                for j in range(i + 1, len(dates_arr)):
                    rd = dates_arr[j]
                    r_z = 0.0 if np.isnan(zscore_arr[j]) else zscore_arr[j]
                    r_pa, r_pb = pa_arr[j], pb_arr[j]
                    
                    out_dates.append(rd)
                    out_pa.append(round(r_pa, 4))
                    out_pb.append(round(r_pb, 4))
                    out_hr.append(1.0)
                    out_z.append(round(float(r_z), 4))
                    out_pos.append(0)
                    out_unrealized.append(0.0)
                    out_realized.append(round(float(state.realized_pnl), 4))
                    out_cum.append(round(float(state.realized_pnl), 4))
                    out_status.append("STOPPED")
                    out_trade_pnl.append(0.0)
                    out_days.append(0)
                    out_delta.append(0.0)
                break 

        if state.position != 0 and out_status:
            last_status = out_status[-1]
            if last_status not in ("EXIT", "STOP_LOSS_TRIGGERED", "PERIOD_END_EXIT", "STOPPED"):
                pnl_before_last_day = out_cum[-2] if len(out_cum) > 1 else 0.0
                
                p_a_last, p_b_last = pa_arr[-1], pb_arr[-1]
                raw_unrealized_final = state.shares_a * (p_a_last - state.entry_price_a) + state.shares_b * (p_b_last - state.entry_price_b)
                exit_fee = (abs(state.shares_a)*p_a_last + abs(state.shares_b)*p_b_last) * self.friction_rate
                
                closed_trade_pnl = raw_unrealized_final - state.trade_entry_fee - exit_fee
                state.realized_pnl += closed_trade_pnl
                daily_delta = state.realized_pnl - pnl_before_last_day 
                
                out_status[-1] = "PERIOD_END_EXIT"
                out_realized[-1] = round(state.realized_pnl, 4)
                out_cum[-1] = round(state.realized_pnl, 4)
                out_unrealized[-1] = 0.0
                out_trade_pnl[-1] = round(closed_trade_pnl, 4)
                out_delta[-1] = round(daily_delta, 4)
                out_days[-1] = state.days_held

        if not out_dates:
            return pd.DataFrame()

        df_out = pd.DataFrame({
            "Date": out_dates, "Price_A": out_pa, "Price_B": out_pb, 
            "Hedge_Ratio": out_hr, "ZScore": out_z, "Position": out_pos, 
            "Unrealized_PnL": out_unrealized, "Realized_PnL": out_realized, 
            "Cumulative_PnL": out_cum, "Status": out_status, 
            "Trade_PnL": out_trade_pnl, "Days_Held": out_days, "Daily_Delta": out_delta
        })
        
        for k, v in base_log.items():
            df_out[k] = v
            
        return df_out

    def run(self, period_start: str, period_end: str) -> tuple:
        """執行該期所有配對的交易模擬"""
        dfs = []
        for _, row in self.selected_pairs.iterrows():
            df_pair = self._simulate_pair(
                period_start=period_start,
                period_end=period_end,
                sector=row.get("Sector", "Unknown"), 
                ticker_a=row["Ticker_A"], 
                ticker_b=row["Ticker_B"], 
                pair_rank=row["Rank"], 
                hedge_ratio=float(row.get("Hedge_Ratio", 1.0)), 
                form_spread_mean=float(row.get("Spread_Mean", 0.0)), 
                form_spread_std=float(row.get("Spread_Std", 1.0)), 
                first_price_a=float(row.get("First_Price_A", 1.0)), 
                first_price_b=float(row.get("First_Price_B", 1.0))
            )
            if not df_pair.empty:
                dfs.append(df_pair)
            
        if not dfs: 
            return pd.DataFrame(), 0.0
            
        log_df = pd.concat(dfs, ignore_index=True)
        period_daily_delta = log_df.groupby("Date")["Daily_Delta"].sum()
        self.period_pnl = float(period_daily_delta.sum()) if not period_daily_delta.empty else 0.0
        
        return log_df, self.period_pnl

# 三、進出場邏輯與部位管理

系統透過一個狀態機 (`PairState`) 追蹤每組配對當前的持倉狀態。

## 1. 進場邏輯 (Entry)

當系統處於空手狀態 (`position = 0`) 時：

- **做空價差 (Enter Short Spread)**：$Z_t > entry\_z$  
  → `position = -1` (賣出 A，買入 B)

- **做多價差 (Enter Long Spread)**：$Z_t < -entry\_z$  
  → `position = 1` (買入 A，賣出 B)

**資金分配**：

$$
Total\_Weight = 1.0 + |\beta| \\
V_A = Capital \times \frac{1.0}{Total\_Weight} \\
V_B = Capital \times \frac{|\beta|}{Total\_Weight}
$$

**進場股數**：

$$
Shares_A = \frac{\pm V_A}{P_{A,t}} \\
Shares_B = \frac{\mp V_B}{P_{B,t}}
$$

## 2. 出場邏輯 (Exit)

**A. 獲利了結 (Normal Exit)**  
- `position == -1` 且 $Z_t \le exit\_z$  
- `position == 1` 且 $Z_t \ge -exit\_z$

**B. 停損出場 (Stop Loss)**  
當 $\frac{-Trade\_PnL}{Capital} \ge stop\_loss\_pct$ 時強制平倉。

**C. 期末強制平倉**：交易期結束時若仍有持倉，則以最後一天收盤價強制平倉。

## 3. 冷卻機制 (Cooldown Mechanism)

停損後記錄 `cooldown_dir`，防止立即反覆進場。需待 Z-Score 回歸至 `exit_z` 才解除冷卻。

In [4]:
# ══════════════════════════════════════════════════════════════════════════════
# Class 3：DataProcessor（數據清理與前置處理模組）
# ══════════════════════════════════════════════════════════════════════════════
class DataProcessor:
    """處理歷史價格載入、產業對應、以及清洗缺失值"""
    def __init__(self, db_path: str, table_name: str = "daily_prices"):
        self.db_path, self.table_name = db_path, table_name

    def load_sector_mapping(self, info_table: str, ticker_col: str = "ticker", sector_col: str = "sector") -> dict:
        try:
            conn = sqlite3.connect(self.db_path)
            df = pd.read_sql_query(f"SELECT {ticker_col}, {sector_col} FROM {info_table}", conn)
            conn.close()
            mapping = {}
            for k, v in zip(df[ticker_col], df[sector_col]):
                if pd.notna(k) and pd.notna(v):
                    mapping[str(k).strip().upper()] = str(v).strip()
            print(f"✅ 成功載入產業分類表 '{info_table}'，共取得 {len(mapping)} 檔標的分類。")
            return mapping
        except Exception as e: 
            print(f"⚠️ [警告] 無法載入產業分類表 '{info_table}'！錯誤原因：{e}")
            print(f"⚠️ 系統將退回「全市場(All_Market)」跨產業配對模式。")
            return {}

    def prepare_backtest_data(self, backtest_start: str, backtest_end: str, formation_window: int):
        conn = sqlite3.connect(self.db_path)
        # 支援 Close 或 Adj_Close
        raw_df = pd.read_sql_query(f"SELECT Date AS date, Symbol AS ticker, COALESCE(Adj_Close, Close) AS price FROM {self.table_name} WHERE COALESCE(Adj_Close, Close) IS NOT NULL ORDER BY Date ASC", conn)
        conn.close()

        raw_df["date"] = pd.to_datetime(raw_df["date"])
        raw_df["price"] = pd.to_numeric(raw_df["price"], errors="coerce")
        raw_df.dropna(subset=["price"], inplace=True)
        raw_df = raw_df[raw_df["price"] > 0]
        
        # 建立 Pivot 價格矩陣
        price_pivot = raw_df.pivot_table(index="date", columns="ticker", values="price", aggfunc="last").sort_index()
        # 移除遺失值超過 20% 的標的，並向前填補最多 5 天
        price_pivot = price_pivot.loc[:, price_pivot.isnull().mean() < 0.20].ffill(limit=5)
        # 移除同日遺失值超過 10% 的日期
        price_pivot.dropna(axis=1, thresh=int(len(price_pivot) * 0.9), inplace=True)
        
        def _safe_parse(d_str, is_end=False):
            if not d_str: return None
            try:
                dt = pd.to_datetime(str(d_str).strip())
                if is_end and len(str(d_str).strip()) == 7:
                    return dt + pd.offsets.MonthEnd(0)
                return dt
            except Exception:
                return None

        bt_start_ts = _safe_parse(backtest_start)
        bt_end_ts = _safe_parse(backtest_end, is_end=True)
        all_dates = price_pivot.index.tolist()

        start_indices = [i for i, d in enumerate(all_dates) if d >= bt_start_ts] if bt_start_ts else []
        first_idx = start_indices[0] if start_indices else 0
        
        # 確保回推足夠的 Formation Window
        data_slice_start = all_dates[max(0, first_idx - formation_window)] if bt_start_ts else price_pivot.index[0]
        data_slice_end = bt_end_ts if bt_end_ts else price_pivot.index[-1]
        price_pivot = price_pivot.loc[data_slice_start:data_slice_end]

        sliced_dates = price_pivot.index.tolist()
        new_start_indices = [i for i, d in enumerate(sliced_dates) if d >= bt_start_ts] if bt_start_ts else []
        local_first_trade_idx = new_start_indices[0] if new_start_indices else formation_window

        return price_pivot, sliced_dates, len(price_pivot), max(local_first_trade_idx, formation_window)


# 四、滾動並行資金配置與部位規模說明

為了科學地管理重疊期間的資金，本系統採用**「平行資金槽 (Slots) 模型」**。以下以 **初始資金 10,000 元** 為例進行由上到下 (Macro to Micro) 的完整剖析。

## 1. 總體資金切割：平行資金槽 (Slots)

系統首先會根據「交易視窗 (Trading Window)」與「滾動步長 (Rolling Step)」來計算**最大並行期數 (Max Concurrent)**，並將總資金等分為數個獨立的資金槽。

**【參數設定範例】**
* 初始總資金 = **10,000 元**
* 交易期長度 (Trading Window) = **126 天** (約半年)
* 滾動步長 (Rolling Step) = **21 天** (約一個月)

**【計算槽數與初始分配】**
* 最大並行期數 = $126 \div 21 = \mathbf{6 \text{ 個平行資金槽}}$
* 每個資金槽的初始資金 = $10,000 \div 6 = \mathbf{1,666.67 \text{ 元}}$

> **系統意義：** 總資金被切成 6 份獨立的「子基金」。第 1 期使用 Slot 1，第 2 期 (21 天後發動) 使用 Slot 2... 直到第 6 期使用 Slot 6。當第 7 期準備發動時，第 1 期剛好結束 126 天的交易，Slot 1 的資金就會被釋放出來給第 7 期使用。這確保了系統永遠有足夠的現金發動新週期的交易。


## 2. 單期配對分配：Top_N 切割 (Capital per Pair)

當某一個期數 (Period) 啟動時，它會從系統接管一個閒置的資金槽。接著，這筆資金會被平均分配給該期篩選出的前 `Top_N` 組最佳配對。

**【參數設定範例】**
* 該期使用的 Slot 資金 = **1,666.67 元**
* 網格參數設定 `Top_N` = **5**

**【計算配對可用資金】**
* 單一配對獲配資金 ($C$) = $1,666.67 \div 5 = \mathbf{333.33 \text{ 元}}$

> **系統意義：** 這代表在該滾動週期內，不論這 5 組配對發出多少次進出場訊號，每組配對每次進場最多只能動用 333.33 元的資金，做到了嚴格的風險分散。

## 3. 單一配對進場：部位規模與對沖比例 (Position Sizing)

當單一配對的 Z-Score 觸發進場訊號時，系統必須將這 333.33 元分配給「股票 A」與「股票 B」來建立中性對沖部位。分配比例取決於形成期算出的**對沖比例 ($\beta$)**。

**【情境設定】**
* 單一配對可用資金 ($C$) = **333.33 元**
* 對沖比例 ($\beta$) = **0.6**
* A 股票價格 ($P_A$) = **50 元** 
* B 股票價格 ($P_B$) = **100 元**
* 訊號：**做空價差 (賣 A 買 B)**

**【計算資金權重】**
* 總權重單位 $Total\_Weight = 1.0 \text{ (屬於 A)} + 0.6 \text{ (屬於 B)} = \mathbf{1.6}$
* A 股票配置資金 ($V_A$) = $333.33 \times (\frac{1.0}{1.6}) = \mathbf{208.33 \text{ 元}}$
* B 股票配置資金 ($V_B$) = $333.33 \times (\frac{0.6}{1.6}) = \mathbf{125.00 \text{ 元}}$
*(檢驗：208.33 + 125.00 = 333.33，資金 100% 運用且無槓桿)*

**【換算實質股數 (Shares)】**
* 賣空 A 股票：$Shares_A = -208.33 \div 50 = \mathbf{-4.166 \text{ 股}}$
* 買入 B 股票：$Shares_B = +125.00 \div 100 = \mathbf{+1.250 \text{ 股}}$

> **系統意義：** 透過 $\beta$ 加權，A 與 B 的曝險市值將達到統計上的波動率中性（A 的價格波動會被 0.6 倍的 B 價格波動所抵銷），將交易風險純粹限縮在「價差收斂」上。

## 4. 週期結算與資金滾動 (Reinvestment & Compounding)

當一個交易期（126 天）結束，系統會強制平倉所有未平倉部位，並結算該期所有配對的總淨利 (Period PnL)。

**【槽位獨立複利機制】**
假設 Slot 1 初始有 1,666.67 元，在經歷 126 天的交易後，該期共淨賺了 **200 元**（已扣除摩擦成本）。
* Slot 1 的期末資金 = $1,666.67 + 200 = \mathbf{1,866.67 \text{ 元}}$

當第 7 期開始並再次借用 Slot 1 時：
* 第 7 期的初始資金將變成 **1,866.67 元**。
* 如果 Top_N 仍為 5，則第 7 期的每一組配對將獲配 $1,866.67 \div 5 = \mathbf{373.33 \text{ 元}}$。

> **系統意義：** 系統具備**「獨立資金槽複利」**特性。虧損的槽位在下一輪能動用的資金會減少（自然降槓桿防爆），獲利的槽位則會加大部位規模。不同槽位之間的資金互不干涉，確保了回測在大型滾動樣本下的穩定性與會計準確性。

## 總結資金流向層級圖

```text
[總資金 $10,000]
   ├── Slot 1 ($1,666) -> 第 1 期 -> 第 7 期 -> 第 13 期 ...
   │      ├── 配對 1 ($333) -> [股票A ($208) | 股票B ($125)]
   │      ├── 配對 2 ($333) -> ...
   │      └── 配對 5 ($333) -> ...
   ├── Slot 2 ($1,666) -> 第 2 期 -> 第 8 期 -> 第 14 期 ...
   ├── Slot 3 ($1,666) -> 第 3 期 -> 第 9 期 ...
   ├── Slot 4 ($1,666) -> 第 4 期 ...
   ├── Slot 5 ($1,666) -> 第 5 期 ...
   └── Slot 6 ($1,666) -> 第 6 期 ...

In [5]:
# ══════════════════════════════════════════════════════════════════════════════
# Class 4：RollingBacktester（滾動回測引擎）
# ══════════════════════════════════════════════════════════════════════════════
class RollingBacktester:
    """負責處理滾動視窗、參數網格搜尋以及交易排程的主引擎"""
    def __init__(self, top_n_list: list, stop_loss_list: list, zscore_window_list: list,
                 entry_z: float, exit_z: float, formation_window: int, trading_window: int, rolling_step: int,
                 fee_rate: float, slippage_rate: float, initial_capital: float,
                 allow_reentry: bool, zscore_clip: float, min_spread_std: float,
                 min_tickers_for_pairing: int, output_dir: Path):
        self.top_n_list = top_n_list
        self.stop_loss_list = stop_loss_list
        self.zscore_window_list = zscore_window_list
        self.entry_z = entry_z
        self.exit_z = exit_z
        self.formation_window = formation_window
        self.trading_window = trading_window
        self.rolling_step = rolling_step
        self.fee_rate = fee_rate
        self.slippage_rate = slippage_rate
        self.initial_capital = initial_capital
        self.allow_reentry = allow_reentry
        self.zscore_clip = zscore_clip
        self.min_spread_std = min_spread_std
        self.min_tickers_for_pairing = min_tickers_for_pairing
        self.output_dir = output_dir

    def run(self, price_pivot: pd.DataFrame, all_dates: list, total_days: int, local_first_trade_idx: int, sector_mapping: dict):
        """執行網格搜索與滾動回測"""
        # 建立網格狀態儲存器與資金槽 (Slots)
        max_concurrent = self.trading_window // self.rolling_step
        states = {}

        for n, sl, z_win in itertools.product(self.top_n_list, self.stop_loss_list, self.zscore_window_list):
            states[(n, sl, z_win)] = {
                "logs": [], 
                "slots": [{"avail_idx": 0, "capital": self.initial_capital / max_concurrent} for _ in range(max_concurrent)]
            }

        roll_start_indices = list(range(local_first_trade_idx, total_days - self.trading_window + 1, self.rolling_step))
        print(f"\n🚀 開始進行 Grid Search，共 {len(roll_start_indices)} 期，每期處理 {len(states)} 種參數組合...")

        # 執行滾動回測主迴圈
        for roll_idx, trade_start_idx in enumerate(roll_start_indices):
            form_start_idx, form_end_idx = trade_start_idx - self.formation_window, trade_start_idx
            trade_end_idx = min(trade_start_idx + self.trading_window, total_days)

            form_data_raw = price_pivot.iloc[form_start_idx:form_end_idx]
            trade_data_raw = price_pivot.iloc[trade_start_idx:trade_end_idx]
            
            # 準備包含 Z-Score 計算所需的延伸歷史資料
            extended_trade_start_idx = max(0, trade_start_idx - max(self.zscore_window_list))
            extended_trade_data_raw = price_pivot.iloc[extended_trade_start_idx:trade_end_idx]
            
            # 過濾在形成期與延伸交易期存在 NaN 的標的，確保滾動計算不會遇到 NaN
            valid_cols = (form_data_raw.isnull().sum() + extended_trade_data_raw.isnull().sum()) == 0
            
            form_data = form_data_raw.loc[:, valid_cols]
            trade_data = trade_data_raw.loc[:, valid_cols]
            trade_dates = trade_data.index
            
            extended_trade_data = extended_trade_data_raw.loc[:, valid_cols]

            if form_data.shape[1] < 2 or trade_data.empty: continue

            trade_start_str, trade_end_str = str(all_dates[trade_start_idx])[:10], str(all_dates[trade_end_idx - 1])[:10]
            form_start_str, form_end_str = str(all_dates[form_start_idx])[:10], str(all_dates[form_end_idx - 1])[:10]
            print(f"  ▶ 處理中：第 {roll_idx+1:02d} 期 (交易: {trade_start_str} ~ {trade_end_str})")

            # 進行配對篩選
            formation = Formation(
                price_df=form_data, 
                form_start=form_start_str, 
                form_end=form_end_str, 
                top_n=max(self.top_n_list), 
                sector_mapping=sector_mapping,
                min_tickers_for_pairing=self.min_tickers_for_pairing
            )
            max_selected_pairs = formation.run()

            if max_selected_pairs.empty: continue

            # 對所有參數組合進行交易模擬
            for n, sl, z_win in itertools.product(self.top_n_list, self.stop_loss_list, self.zscore_window_list):
                selected_pairs = max_selected_pairs.head(n)
                state = states[(n, sl, z_win)]
                slots = state["slots"]
                
                # 分配可用資金槽
                free_slots = [i for i, s in enumerate(slots) if s["avail_idx"] <= trade_start_idx]
                if free_slots:
                    slot_idx = free_slots[0]
                else:
                    slot_idx = min(range(max_concurrent), key=lambda i: slots[i]["avail_idx"])

                current_period_capital = slots[slot_idx]["capital"]
                current_capital_per_pair = current_period_capital / n

                trading = Trading(
                    price_df=extended_trade_data,
                    trade_dates=trade_dates,
                    selected_pairs=selected_pairs,
                    capital_per_pair=current_capital_per_pair,
                    fee_rate=self.fee_rate,
                    slippage_rate=self.slippage_rate,
                    stop_loss_pct=sl,
                    entry_z=self.entry_z,
                    exit_z=self.exit_z,
                    zscore_window=z_win,
                    allow_reentry=self.allow_reentry,
                    zscore_clip=self.zscore_clip,
                    min_spread_std=self.min_spread_std
                )
                
                trade_log_df, period_pnl = trading.run(trade_start_str, trade_end_str)
                
                if not trade_log_df.empty:
                    state["logs"].append(trade_log_df)
                    
                slots[slot_idx]["capital"] = max(0, current_period_capital + period_pnl)
                slots[slot_idx]["avail_idx"] = trade_end_idx

        # 迴圈結束後呼叫匯出功能
        self._export_results(states)

    def _export_results(self, states: dict):
        """將每種參數組合的紀錄匯出為獨立 CSV"""
        print("\n✅ 回測完成！正在匯出交易紀錄檔案...")
        for (n, sl, z_win), state in states.items():
            if state["logs"]:
                full_log_df = pd.concat(state["logs"], ignore_index=True)
                sl_str = f"SL{int(sl*100)}" if sl > 0 else "SL0"
                filename = f"TradeLogs_Top{n}_{sl_str}_ZWin{z_win}.csv"
                filepath = self.output_dir / filename
                full_log_df.to_csv(filepath, index=False)
                print(f"  - 已輸出: {filename} (共 {len(full_log_df)} 筆紀錄)")
                
        print(f"\n📁 所有交易紀錄已成功儲存至: {self.output_dir}")

# 五、損益計算與會計原則 (Mark-to-Market)

系統採用每日市價評估 (Mark-to-Market) 原則：

- **空手狀態**：$Unrealized\_PnL = 0$
- **持倉期間**：$Unrealized\_PnL =$ 當前交易淨損益（已扣除進場費與預估出場費）
- **平倉當日**：將 $Trade\_PnL$ 計入 $Realized\_PnL$

每日淨值變化 (`Daily_Delta`) 加總所有配對後，用於更新下一期的可用資金。

In [7]:
# ══════════════════════════════════════════════════════════════════════════════
# 主程式：自動參數網格搜尋 
# ══════════════════════════════════════════════════════════════════════════════
if __name__ == "__main__":
    
    # --- 1. 基本參數與路徑設定 ---
    MIN_TICKERS_FOR_PAIRING = 2  
    ZSCORE_CLIP = 10.0           
    MIN_SPREAD_STD = 1e-6

    # 確保您有此相對路徑下的 db 檔案
    DB_PATH, TABLE_NAME = r"../data/sp500_Current.db", "Daily_Prices"
    BACKTEST_START, BACKTEST_END = "2000-01", "2025-12"
    INFO_TABLE_NAME, TICKER_COL_NAME, SECTOR_COL_NAME = "Constituents", "Symbol", "GICS_Sector"

    ALLOW_REENTRY = True            # 是否允許再進場
    OUTPUT_DIR = Path(r"../results/current/SSD_basic_ReEntry")
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    
    # --- 2. 網格搜尋參數設定 ---
    TOP_N_LIST = [1, 5, 20]
    STOP_LOSS_LIST = [0, 0.05, 0.15]      # 0 表示不停損
    ZSCORE_WINDOW_LIST = [0, 20, 60]      # 0 表示固定 Z 值
    
    ENTRY_Z, EXIT_Z = 2.0, 0.0
    FORMATION_WINDOW, TRADING_WINDOW, ROLLING_STEP = 252, 126, 21
    
    # --- 3. 交易成本與環境設定 ---
    FEE_RATE = 0.001
    SLIPPAGE_RATE = 0.001 
    INITIAL_CAPITAL = 10000 
    USE_SECTOR_PAIRING = True       # 是否使用產業分類
        
    # --- 4. 資料前處理 ---
    processor = DataProcessor(db_path=DB_PATH, table_name=TABLE_NAME)
    
    if USE_SECTOR_PAIRING:
        sector_mapping = processor.load_sector_mapping(INFO_TABLE_NAME, TICKER_COL_NAME, SECTOR_COL_NAME)
    else:
        sector_mapping = {}
        print("ℹ️ 產業分類配對已關閉 (USE_SECTOR_PAIRING = False)，系統將進行全市場全標的配對。")
        
    # 執行資料載入與格式化
    try:
        price_pivot, all_dates, total_days, local_first_trade_idx = processor.prepare_backtest_data(BACKTEST_START, BACKTEST_END, FORMATION_WINDOW)

        # --- 5. 啟動回測引擎 ---
        # 將所有參數送入新建立的 RollingBacktester，並呼叫 run()
        engine = RollingBacktester(
            top_n_list=TOP_N_LIST, stop_loss_list=STOP_LOSS_LIST, zscore_window_list=ZSCORE_WINDOW_LIST,
            entry_z=ENTRY_Z, exit_z=EXIT_Z, formation_window=FORMATION_WINDOW,
            trading_window=TRADING_WINDOW, rolling_step=ROLLING_STEP, fee_rate=FEE_RATE,
            slippage_rate=SLIPPAGE_RATE, initial_capital=INITIAL_CAPITAL, allow_reentry=ALLOW_REENTRY,
            zscore_clip=ZSCORE_CLIP, min_spread_std=MIN_SPREAD_STD,
            min_tickers_for_pairing=MIN_TICKERS_FOR_PAIRING, output_dir=OUTPUT_DIR
        )
        
        engine.run(price_pivot, all_dates, total_days, local_first_trade_idx, sector_mapping)

    except Exception as e:
        print(f"\n❌ 執行發生錯誤：{e}")
        print("💡 提示: 請確認資料庫 `../data/sp500.db` 存在，且資料表格式符合預期。")

✅ 成功載入產業分類表 'Constituents'，共取得 503 檔標的分類。

🚀 開始進行 Grid Search，共 294 期，每期處理 27 種參數組合...
  ▶ 處理中：第 01 期 (交易: 2001-01-02 ~ 2001-07-02)
  ▶ 處理中：第 02 期 (交易: 2001-02-01 ~ 2001-08-01)
  ▶ 處理中：第 03 期 (交易: 2001-03-05 ~ 2001-08-30)
  ▶ 處理中：第 04 期 (交易: 2001-04-03 ~ 2001-10-05)
  ▶ 處理中：第 05 期 (交易: 2001-05-03 ~ 2001-11-05)
  ▶ 處理中：第 06 期 (交易: 2001-06-04 ~ 2001-12-05)
  ▶ 處理中：第 07 期 (交易: 2001-07-03 ~ 2002-01-07)
  ▶ 處理中：第 08 期 (交易: 2001-08-02 ~ 2002-02-06)
  ▶ 處理中：第 09 期 (交易: 2001-08-31 ~ 2002-03-08)
  ▶ 處理中：第 10 期 (交易: 2001-10-08 ~ 2002-04-09)
  ▶ 處理中：第 11 期 (交易: 2001-11-06 ~ 2002-05-08)
  ▶ 處理中：第 12 期 (交易: 2001-12-06 ~ 2002-06-07)
  ▶ 處理中：第 13 期 (交易: 2002-01-08 ~ 2002-07-09)
  ▶ 處理中：第 14 期 (交易: 2002-02-07 ~ 2002-08-07)
  ▶ 處理中：第 15 期 (交易: 2002-03-11 ~ 2002-09-06)
  ▶ 處理中：第 16 期 (交易: 2002-04-10 ~ 2002-10-07)
  ▶ 處理中：第 17 期 (交易: 2002-05-09 ~ 2002-11-05)
  ▶ 處理中：第 18 期 (交易: 2002-06-10 ~ 2002-12-05)
  ▶ 處理中：第 19 期 (交易: 2002-07-10 ~ 2003-01-07)
  ▶ 處理中：第 20 期 (交易: 2002-08-08 ~ 2003-02-06)
  ▶ 處理中：第 21 

# 六、績效計算邏輯與公式說明

## 1. 財務報酬指標

### 1.1 最終淨值 (Final Equity) 與 總損益 (Total PnL)

* **邏輯**：將該策略每日的所有配對損益 (`Daily_Delta`) 進行加總，得到每日投資組合總損益。將其累加得到累計損益，再加上初始本金即為淨值。

* **公式**：

  * $Daily\_Portfolio\_Delta_t = \sum Daily\_Delta_{i,t}$

  * $Total\_PnL = \sum_{t=1}^{T} Daily\_Portfolio\_Delta_t$

  * $Final\_Equity = INITIAL\_CAPITAL + Total\_PnL$

### 1.2 累積報酬率 (Cumulative Return)

* **邏輯**：程式碼先將每日淨值 (`Equity`) 重新取樣（Resample）為「月底最後一日淨值」，計算出「每月報酬率 ($R_m$)」，再將所有月份的報酬率進行連乘。

* **公式**：

  * $R_m = \frac{Equity_m - Equity_{m-1}}{Equity_{m-1}}$

  * $Cumulative\_Return = \prod_{m=1}^{M} (1 + R_m) - 1$

### 1.3 年化報酬率 (Annualized Return)

* **邏輯**：基於上述計算出的累積報酬率與總月數 ($M$)，推算成年化標準。

* **公式**：

  * $Annualized\_Return = (1 + Cumulative\_Return)^{\frac{12}{M}} - 1$

### 1.4 夏普值 (Sharpe Ratio)

* **邏輯**：衡量每單位風險所獲得的超額報酬。此處的程式碼邏輯將「每日報酬」定義為「每日損益除以**初始本金**」，並採用一年 252 個交易日進行年化，無風險利率預設為 0。

* **公式**：

  * $r_t = \frac{Daily\_Portfolio\_Delta_t}{INITIAL\_CAPITAL}$

  * $Sharpe\_Ratio = \sqrt{252} \times \frac{Mean(r_t)}{Std(r_t)}$

### 1.5 最大回撤 (Maximum Drawdown, MDD)

* **邏輯**：計算累計損益 (`Cumulative_PnL`) 從歷史最高點以來的最大跌幅。分母固定使用初始本金。

* **公式**：

  * $Roll\_Max_t = \max_{k=1}^{t} (Cumulative\_PnL_k)$

  * $Drawdown_t = Cumulative\_PnL_t - Roll\_Max_t$

  * $MDD = \frac{\min(Drawdown)}{INITIAL\_CAPITAL}$

## 2. 資本效率指標

### 2.1 承諾資本報酬率 (Return on Committed Capital, RCC)

* **邏輯**：評估分配給該策略的總承諾資本 ($c\_period$) 所產生的報酬。在目前的程式設定中，承諾資本等於初始本金。

* **公式**：

  * $c\_period = INITIAL\_CAPITAL$

  * $RCC = \frac{Total\_PnL}{c\_period}$

### 2.2 實際投入資本報酬率 (Return on Engaged Capital, REC)

* **邏輯**：評估「實際有被動用到」的資金所產生的報酬。首先計算單一配對所分配到的本金 ($c\_pair$)，再乘上該策略在回測期間「實際交易過的獨立配對數量 ($N_{traded}$ )」。

* **公式**：

  * $Top\_N\_Int =$ 策略參數設定之 Top N 數量 (如 5, 20)

  * $c\_pair = \frac{c\_period}{Top\_N\_Int}$

  * $Engaged\_Capital = N_{traded} \times c\_pair$

  * $REC = \frac{Total\_PnL}{Engaged\_Capital}$

## 3. 交易次數統計邏輯

此部分利用 `Prev_Pos = shift(1)` 偵測狀態改變 (`Direction_Change`) 來計算客觀次數。

* **出場總次數 (Exits Total)**：發生狀態改變且前一日不為 0（持有部位），包含平倉與換向。

  * 條件：`Position != Prev_Pos` 且 `Prev_Pos != 0`

* **停損次數 (Stop Losses)**：在上述出場條件成立的日子，其 `Status` 欄位包含 `stop`, `sl`, 或 `停損`。

* **正常平倉次數 (Normal Exits)**：

  * 公式：$Normal\_Exits = Exits\_Total - Stop\_Losses$

* **強制平倉次數 (Forced Closes)**：回測資料最後一筆紀錄中，`Position` 不等於 0 的數量。

* **進場次數 (Entries)**：

  * 公式：$Entries = Exits\_Total + Forced\_Closes$

## 4. 淨損益 (Gross Profit / Loss) 狀態機邏輯

為了精確計算每「單筆交易 (Trade Cycle)」的盈虧，避免同一筆交易的漲跌日被拆分，程式採用狀態機 (State Machine) 進行分組：

1. **狀態 ID 生成 (`State_ID`)**：利用 `Position != Prev_Pos` 來判定狀態是否改變。當狀態改變時，`State_ID` 累加 1。

2. **損益歸屬 (`Prev_State_ID`)**：將 `State_ID` 向下遞延一日。此舉確保「平倉日當天產生的結算損益」會與「前一日的持倉狀態」綁定在同一個 ID 內。

3. **單筆交易損益彙整**：針對 `Prev_Pos != 0` 或 `Daily_Delta != 0` 的資料，依據 `Ticker_A`, `Ticker_B`, `Prev_State_ID` 進行群組加總，得出每一次獨立交易的總損益 (`Trade_PnL_sum`)。

4. **淨收益與淨損失分類**：

   * $Gross\_Profit = \sum Trade\_PnL\_sum \quad (當 \ Trade\_PnL\_sum > 0)$

   * $Gross\_Loss = \sum Trade\_PnL\_sum \quad (當 \ Trade\_PnL\_sum < 0)$